In [ ]:
embedding = torch.nn.Embedding(
    num_embeddings,
    embedding_dim,
    padding_idx = None,
    norm_type=2.0
)

## 라이브러리 다운

In [1]:
!pip install Korpora

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.8/57.8 kB 6.4 MB/s eta 0:00:00


In [1]:
!pip install konlpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.5/438.5 kB 21.5 MB/s eta 0:00:00


## 6.3 기본 skip_gram 클래스

In [2]:
from torch import nn

class VanillaSkipgram(nn.Module):
  def __init__(self, vocab_size, embedding_dim):
    super().__init__()
    self.embedding = nn.Embedding(
        num_embeddings = vocab_size,
        embedding_dim = embedding_dim
    )
    self.linear = nn.Linear(
        in_features = embedding_dim,
        out_features = vocab_size
    )
  def forward(self, input_ids):
    embeddings = self.embedding(input_ids)
    output = self.linear(embeddings)
    return output

## 6.4 영화 리뷰 데이터세트 전처리

In [3]:
import pandas as pd
from Korpora import Korpora
from konlpy.tag import Okt

corpus = Korpora.load("nsmc")
corpus = pd.DataFrame(corpus.test)

tokenizer = Okt()
tokens = [tokenizer.morphs(review) for review in corpus.text]
print(tokens[:3])


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : e9t@github
    Repository : https://github.com/e9t/nsmc
    References : www.lucypark.kr/docs/2015-pyconkr/#39

    Naver sentiment movie corpus v1.0
    This is a movie review dataset in the Korean language.
    Reviews were scraped from Naver Movies.

    The dataset construction is based on the method noted in
    [Large movie review dataset][^1] from Maas et al., 2011.

    [^1]: http://ai.stanford.edu/~amaas/data/sentiment/

    # License
    CC0 1.0 Universal (CC0 1.0) Public Domain Dedication
    Details in https://creativecommons.org/publicdomain/zero/1.0/



[nsmc] download ratings_train.txt: 14.6MB [00:00, 106MB/s]                             
[nsmc] download ratings_test.txt: 4.90MB [00:00, 57.8MB/s]


[['굳', 'ㅋ'], ['GDNTOPCLASSINTHECLUB'], ['뭐', '야', '이', '평점', '들', '은', '....', '나쁘진', '않지만', '10', '점', '짜', '리', '는', '더', '더욱', '아니잖아']]


##6.5 단어 사전 구축

In [4]:
from collections import Counter


def build_vocab(corpus, n_vocab, special_tokens):
    counter = Counter()
    for tokens in corpus:
        counter.update(tokens)
    vocab = special_tokens
    for token, count in counter.most_common(n_vocab):
        vocab.append(token)
    return vocab


vocab = build_vocab(corpus=tokens, n_vocab=5000, special_tokens=[""])
token_to_id = {token: idx for idx, token in enumerate(vocab)}
id_to_token = {idx: token for idx, token in enumerate(vocab)}

print(vocab[:10])
print(len(vocab))

['', '.', '이', '영화', '의', '..', '가', '에', '...', '을']
5001


## 6.6 Skip-gram의 단어 쌍 추출

In [5]:
def get_word_pairs(tokens, window_size):
    pairs = []
    for sentence in tokens:
        sentence_length = len(sentence)
        for idx, center_word in enumerate(sentence):
            window_start = max(0, idx - window_size)
            window_end = min(sentence_length, idx + window_size + 1)
            center_word = sentence[idx]
            context_words = sentence[window_start:idx] + sentence[idx+1:window_end]
            for context_word in context_words:
                pairs.append([center_word, context_word])
    return pairs


word_pairs = get_word_pairs(tokens, window_size=2)
print(word_pairs[:5])

[['굳', 'ㅋ'], ['ㅋ', '굳'], ['뭐', '야'], ['뭐', '이'], ['야', '뭐']]


## 6.7 인덱스 쌍 변환

In [6]:
def get_index_pairs(word_pairs, token_to_id):
    pairs = []
    unk_index = token_to_id[""]
    for word_pair in word_pairs:
        center_word, context_word = word_pair
        center_index = token_to_id.get(center_word, unk_index)
        context_index = token_to_id.get(context_word, unk_index)
        pairs.append([center_index, context_index])
    return pairs


index_pairs = get_index_pairs(word_pairs, token_to_id)
print(index_pairs[:5])
print(len(vocab))

[[595, 100], [100, 595], [77, 176], [77, 2], [176, 77]]
5001


## 6.8 데이터로더 적용

In [7]:
import torch
from torch.utils.data import TensorDataset, DataLoader


index_pairs = torch.tensor(index_pairs)
center_indexes = index_pairs[:, 0]
context_indexes = index_pairs[:, 1]

dataset = TensorDataset(center_indexes, context_indexes)
dataloader = DataLoader(dataset, batch_size=512, shuffle=True)

##6.9 Skip-gram 모델 준비 작업

In [8]:
from torch import optim


device = "cuda" if torch.cuda.is_available() else "cpu"
word2vec = VanillaSkipgram(vocab_size=len(token_to_id), embedding_dim=128).to(device)
criterion = nn.CrossEntropyLoss().to(device)
optimizer = optim.SGD(word2vec.parameters(), lr=0.1)

In [9]:
print(device)

cuda


## 6.10 모델 학습

In [10]:
for epoch in range(10):
    cost = 0.0
    for input_ids, target_ids in dataloader:
        input_ids = input_ids.to(device)
        target_ids = target_ids.to(device)

        logits = word2vec(input_ids)
        loss = criterion(logits, target_ids)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        cost += loss

    cost = cost / len(dataloader)
    print(f"Epoch : {epoch+1:4d}, Cost : {cost:.3f}")

Epoch :    1, Cost : 6.868
Epoch :    2, Cost : 6.359
Epoch :    3, Cost : 6.220
Epoch :    4, Cost : 6.147
Epoch :    5, Cost : 6.098
Epoch :    6, Cost : 6.064
Epoch :    7, Cost : 6.037
Epoch :    8, Cost : 6.016
Epoch :    9, Cost : 5.998
Epoch :   10, Cost : 5.983


## 6.11 임베딩 값 추출

In [11]:
token_to_embedding = dict()
embedding_matrix = word2vec.embedding.weight.detach().cpu().numpy()

for word, embedding in zip(vocab, embedding_matrix):
    token_to_embedding[word] = embedding

index = 30
token = vocab[index]
token_embedding = token_to_embedding[token]
print(token)
print(token_embedding)

연기
[-1.43491185e+00  1.85612738e-01 -1.38496161e+00  3.15398514e-01
  1.51383388e+00 -7.43785977e-01 -1.60658523e-01  7.77954832e-02
 -7.66476750e-01  7.81607807e-01 -4.86628443e-01  1.09908737e-01
  6.21469140e-01  1.24043846e+00 -1.10366917e+00  4.54501241e-01
  1.58930337e+00  6.91349745e-01 -1.54758632e+00  3.02544713e-01
  1.04968749e-01  8.12935084e-02  1.49603391e+00  7.35619545e-01
  7.33853459e-01  9.86465871e-01  7.86084950e-01 -6.40624985e-02
 -1.48589599e+00  1.01343179e+00 -4.55971286e-02  3.87782156e-01
  5.09078085e-01 -3.79643738e-01 -6.32986367e-01 -7.64517605e-01
  7.40570068e-01 -1.55949473e+00  1.57361832e-02 -6.35026515e-01
  1.38169318e-01 -1.54084587e+00  1.70307624e+00  1.13024342e+00
  1.15398097e+00 -6.64159834e-01  5.80670536e-01 -1.20960128e+00
 -6.81458414e-01  5.76044321e-01  5.71790695e-01 -8.14529181e-01
 -9.51589763e-01 -4.74708200e-01 -1.50942957e+00  7.83014596e-01
  7.82204093e-04  6.02433622e-01  2.46186042e+00 -1.30336034e+00
 -6.80156052e-01  8.01

## 6.12 단어 임베딩 유사도 계산

In [12]:
import numpy as np
from numpy.linalg import norm


def cosine_similarity(a, b):
    cosine = np.dot(b, a) / (norm(b, axis=1) * norm(a))
    return cosine

def top_n_index(cosine_matrix, n):
    closest_indexes = cosine_matrix.argsort()[::-1]
    top_n = closest_indexes[1 : n + 1]
    return top_n


cosine_matrix = cosine_similarity(token_embedding, embedding_matrix)
top_n = top_n_index(cosine_matrix, n=5)

print(f"{token}와 가장 유사한 5 개 단어")
for index in top_n:
    print(f"{id_to_token[index]} - 유사도 : {cosine_matrix[index]:.4f}")

연기와 가장 유사한 5 개 단어
고인 - 유사도 : 0.3425
시리즈 - 유사도 : 0.3392
b - 유사도 : 0.2929
귀여워요 - 유사도 : 0.2613
였으면 - 유사도 : 0.2545


# Gensim Model 실습

In [13]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 77.5 MB/s eta 0:00:00


In [14]:
import pandas as pd
from Korpora import Korpora
from konlpy.tag import Okt


corpus = Korpora.load("nsmc")
corpus = pd.DataFrame(corpus.test)


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : e9t@github
    Repository : https://github.com/e9t/nsmc
    References : www.lucypark.kr/docs/2015-pyconkr/#39

    Naver sentiment movie corpus v1.0
    This is a movie review dataset in the Korean language.
    Reviews were scraped from Naver Movies.

    The dataset construction is based on the method noted in
    [Large movie review dataset][^1] from Maas et al., 2011.

    [^1]: http://ai.stanford.edu/~amaas/data/sentiment/

    # License
    CC0 1.0 Universal (CC0 1.0) Public Domain Dedication
    Details in https://creativecommons.org/publicdomain/zero/1.0/

[Korpora] Corpus `nsmc` is already installed at /root/Korpora/nsmc/ratings_train.txt
[Korpora] Corpus `nsmc` is already installed at /root/Korpora/nsmc/ra

In [ ]:
tokenizer = Okt()
tokens = [tokenizer.morphs(review) for review in corpus.text]

## 6.13 Word2Vec 모델 학습

In [16]:
from gensim.models import Word2Vec


word2vec = Word2Vec(
    sentences=tokens,
    vector_size=128,
    window=5,
    min_count=1,
    sg=1,
    epochs=3,
    max_final_vocab=10000
)

## 6.14 임베딩 추출 및 유사도 계산

In [17]:
word = "연기"
print(word2vec.wv[word])
print(word2vec.wv.most_similar(word, topn=5))
print(word2vec.wv.similarity(w1=word, w2="연기력"))

[ 5.66337854e-02 -4.24261063e-01  3.32563907e-01  5.62338531e-01
  3.73965725e-02 -5.04531823e-02  2.13160664e-01  1.61866862e-02
 -5.69112837e-01  2.40636885e-01 -1.92841828e-01 -2.34378487e-01
 -4.47112508e-02 -9.19380561e-02 -2.29645804e-01 -3.83755006e-02
 -1.31947234e-01  3.51831466e-01 -1.98000595e-01  4.43950266e-01
  4.95736331e-01  6.41559482e-01 -7.40892291e-02  5.16703054e-02
 -1.43122273e-02  6.57055601e-02 -4.22586918e-01 -1.17206991e-01
  2.31791958e-01 -8.27265903e-02 -4.35987979e-01  2.20670193e-01
  2.73571342e-01 -1.29116371e-01 -2.61146784e-01 -2.06510082e-01
  1.63669705e-01 -4.02150691e-01 -7.52960816e-02 -4.13615435e-01
 -1.63945407e-02  1.95925534e-01 -2.01207444e-01 -4.84238923e-01
 -2.10708290e-01  2.54934937e-01 -3.04749578e-01 -3.46173525e-01
  5.72677217e-02  1.84967563e-01  8.19546461e-01  3.73309672e-01
  7.67229646e-02  1.93164214e-01 -4.63751793e-01 -5.42181909e-01
  2.02075303e-01  2.07777247e-01 -1.63178280e-01 -8.28281119e-02
 -1.61899805e-01 -1.46611